# PHASE 6B — LOCKED SUBJECT-DISJOINT REPLICATION

**Primary hypothesis**

At equal intervention budget \(k=2\), the interaction-aware circuit
`[(10,"Q"), (8,"S")]` achieves greater normalized causal recovery than
the first-order AIE circuit `[(10,"Q"), (9,"Q")]`.

**Primary endpoint**

\[
\Delta = \mathrm{recovery}_{INT}-\mathrm{recovery}_{AIE}.
\]

**Success criteria fixed before evaluation**

- bootstrap paired 95% CI lower bound \(>0\);
- practical effect: mean \(\Delta \ge 0.02\).

**Primary corruption:** Gaussian subject-embedding corruption, scale 3.0.

**Robustness:** scales 2.5, 3.0, 3.5; seeds 101, 202, 303.

**Freshness rule:** exclude every subject seen in Phase 5 or Phase 6A, and require 128 unique subjects inside Phase 6B itself.

Do not change sites, thresholds, noise scales, seeds, endpoint, or success criteria after running the benchmark.


In [1]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [2]:
MODEL_ID = "openai-community/gpt2"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    local_files_only=True
).to(DEVICE).eval()

N_LAYERS = model.config.n_layer

EMBED_STD = (
    model.transformer.wte.weight
    .detach()
    .std()
    .item()
)

print("layers:", N_LAYERS)
print("embedding std:", EMBED_STD)

device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

layers: 12
embedding std: 0.14369554817676544


In [3]:
counterfact = load_dataset(
    "azhx/counterfact",
    split="test"
)

print(counterfact)


Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 2191
})


In [4]:
# FROZEN SETTINGS

AIE_PAIR = [
    (10, "Q"),
    (9,  "Q"),
]

INTERACTION_PAIR = [
    (10, "Q"),
    (8,  "S"),
]

PRIMARY_SCALE = 3.0

NOISE_SCALES = [2.5, 3.0, 3.5]
NOISE_SEEDS = [101, 202, 303]

N_FRESH = 128

REFERENCE_SEED = 424242
REFERENCE_SCALE = 3.0

MIN_CLEAN_MARGIN = 1.0
MIN_REFERENCE_TE = 1.0
MIN_EVAL_TE = 0.5

SAMPLING_SEED = 20260910

PHASE5_SEEN_PATH = "phase5_seen_facts.csv"
PHASE6A_MANIFEST_PATH = "phase6_fresh_manifest.csv"

MANIFEST_V2 = "phase6b_locked_manifest.csv"
PROTOCOL_V2 = "phase6b_locked_protocol.json"
RESULT_PATH = "phase6b_locked_results.csv"


In [5]:
# EXCLUDE ALL PREVIOUSLY OBSERVED SUBJECTS

phase5_seen = pd.read_csv(PHASE5_SEEN_PATH)
phase6a_manifest = pd.read_csv(PHASE6A_MANIFEST_PATH)

def norm_subject(x):
    return str(x).strip().lower()

PHASE5_SUBJECTS = set(
    phase5_seen["subject"].map(norm_subject)
)

PHASE6A_SUBJECTS = set(
    phase6a_manifest["subject"].map(norm_subject)
)

EXCLUDED_SUBJECTS = PHASE5_SUBJECTS | PHASE6A_SUBJECTS

print("Phase-5 subjects:", len(PHASE5_SUBJECTS))
print("Phase-6A subjects:", len(PHASE6A_SUBJECTS))
print("Overlap Phase5/Phase6A:", len(PHASE5_SUBJECTS & PHASE6A_SUBJECTS))
print("Total excluded subjects:", len(EXCLUDED_SUBJECTS))

assert len(PHASE5_SUBJECTS) == 64
assert len(PHASE6A_SUBJECTS) == 128


Phase-5 subjects: 64
Phase-6A subjects: 128
Overlap Phase5/Phase6A: 6
Total excluded subjects: 186


In [6]:
def one_token_id(text):
    ids = tokenizer.encode(
        " " + text.strip(),
        add_special_tokens=False
    )
    return ids[0] if len(ids) == 1 else None


def subject_token_positions(prompt, subject):
    enc = tokenizer(
        prompt,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    start = prompt.find(subject)
    if start < 0:
        return []

    end = start + len(subject)
    positions = []

    for i, (a, b) in enumerate(enc["offset_mapping"]):
        if b > start and a < end:
            positions.append(i)

    return positions


def logit_margin(logits, true_id, alt_id):
    return (
        logits[0, -1, true_id]
        - logits[0, -1, alt_id]
    ).item()


In [7]:
def corrupted_embeddings(
    input_ids,
    subj_positions,
    seed,
    noise_multiplier
):
    embeds = (
        model.transformer.wte(input_ids)
        .detach()
        .clone()
    )

    g = torch.Generator(device=embeds.device)
    g.manual_seed(int(seed))

    noise = torch.randn(
        embeds.shape,
        generator=g,
        device=embeds.device,
        dtype=embeds.dtype
    )

    noise *= noise_multiplier * EMBED_STD

    for pos in subj_positions:
        embeds[:, pos, :] += noise[:, pos, :]

    return embeds


In [8]:
def prepare_fresh_case(record, dataset_index):
    rr = record["requested_rewrite"]

    subject = rr["subject"]
    prompt = rr["prompt"].format(subject)

    true_text = rr["target_true"]["str"]
    new_text = rr["target_new"]["str"]

    true_id = one_token_id(true_text)
    new_id = one_token_id(new_text)

    if true_id is None or new_id is None or true_id == new_id:
        return None

    subj_positions = subject_token_positions(prompt, subject)
    if not subj_positions:
        return None

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.inference_mode():
        clean_out = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    D_clean = logit_margin(
        clean_out.logits,
        true_id,
        new_id
    )

    if D_clean < MIN_CLEAN_MARGIN:
        return None

    ref_emb = corrupted_embeddings(
        input_ids,
        subj_positions,
        REFERENCE_SEED + int(dataset_index),
        REFERENCE_SCALE
    )

    with torch.inference_mode():
        ref_out = model(
            inputs_embeds=ref_emb,
            attention_mask=attention_mask
        )

    D_ref = logit_margin(
        ref_out.logits,
        true_id,
        new_id
    )

    TE_ref = D_clean - D_ref

    if TE_ref < MIN_REFERENCE_TE:
        return None

    return {
        "case_id": int(record["case_id"]),
        "dataset_index": int(dataset_index),
        "prompt": prompt,
        "subject": subject,
        "true_text": true_text,
        "new_text": new_text,
        "true_id": int(true_id),
        "new_id": int(new_id),
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "subject_positions": subj_positions,
        "subject_last": subj_positions[-1],
        "query_pos": input_ids.shape[1] - 1,
        "D_clean": float(D_clean),
        "TE_reference": float(TE_ref),
    }


In [9]:
# BUILD PHASE 6B HOLDOUT
# Important: unique subject both relative to earlier phases AND within Phase 6B.

rng = np.random.default_rng(SAMPLING_SEED)
indices = rng.permutation(len(counterfact))

fresh_cases_v2 = []
selected_subjects = set()

for idx in indices:
    record = counterfact[int(idx)]

    subject_key = norm_subject(
        record["requested_rewrite"]["subject"]
    )

    if subject_key in EXCLUDED_SUBJECTS:
        continue

    if subject_key in selected_subjects:
        continue

    case = prepare_fresh_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    fresh_cases_v2.append(case)
    selected_subjects.add(subject_key)

    print(
        len(fresh_cases_v2),
        case["subject"],
        "| reference TE =",
        round(case["TE_reference"], 3)
    )

    if len(fresh_cases_v2) >= N_FRESH:
        break


1 Ring 2 | reference TE = 1.504
2 Portobello, Edinburgh | reference TE = 5.71
3 Mariano Gonzalvo | reference TE = 2.677
4 Alain de Benoist | reference TE = 2.98
5 Black Sparrow Books | reference TE = 1.306
6 Margaret Walker | reference TE = 2.006
7 Jimmy Choo | reference TE = 3.421
8 Carnival of Venice | reference TE = 9.93
9 Bavaria | reference TE = 1.14
10 Guillaumes | reference TE = 1.645
11 Johann Andreas Schmeller | reference TE = 5.039
12 Obsidian Entertainment | reference TE = 4.125
13 Il Gazzettino | reference TE = 2.327
14 Alain Mabanckou | reference TE = 4.218
15 Jules Paul Benjamin Delessert | reference TE = 1.274
16 Stevie Wonder | reference TE = 2.987
17 Sous le vent | reference TE = 1.319
18 Berlin International Film Festival | reference TE = 9.268
19 Dmitri Nabokov | reference TE = 1.016
20 Claudio Abbado | reference TE = 3.329
21 Acura Legend | reference TE = 4.726
22 Jean-Michel Jarre | reference TE = 1.586
23 Heilmann & Littmann | reference TE = 3.068
24 Autonomous Un

In [10]:
# HARD AUDIT BEFORE SAVING

assert len(fresh_cases_v2) == N_FRESH

new_subjects = {
    norm_subject(c["subject"])
    for c in fresh_cases_v2
}

assert len(new_subjects) == N_FRESH
assert (new_subjects & EXCLUDED_SUBJECTS) == set()

new_case_ids = [c["case_id"] for c in fresh_cases_v2]
assert len(set(new_case_ids)) == N_FRESH

print("Fresh Phase-6B cases:", len(fresh_cases_v2))
print("Unique Phase-6B subjects:", len(new_subjects))
print("Overlap with prior phases:", len(new_subjects & EXCLUDED_SUBJECTS))


Fresh Phase-6B cases: 128
Unique Phase-6B subjects: 128
Overlap with prior phases: 0


In [11]:
# SAVE AND HASH THE LOCKED MANIFEST

fresh_manifest_v2 = pd.DataFrame([
    {
        "case_id": c["case_id"],
        "dataset_index": c["dataset_index"],
        "subject": c["subject"],
        "prompt": c["prompt"],
        "true_text": c["true_text"],
        "new_text": c["new_text"],
        "D_clean": c["D_clean"],
        "TE_reference": c["TE_reference"],
    }
    for c in fresh_cases_v2
])

fresh_manifest_v2.to_csv(
    MANIFEST_V2,
    index=False
)

with open(MANIFEST_V2, "rb") as f:
    MANIFEST_SHA256 = hashlib.sha256(f.read()).hexdigest()

print("manifest:", MANIFEST_V2)
print("sha256:", MANIFEST_SHA256)
fresh_manifest_v2.head()


manifest: phase6b_locked_manifest.csv
sha256: db18afb7d278318662089ab688dd97230acc5c378169cbba143732997646d10f


,case_id,dataset_index,subject,prompt,true_text,new_text,D_clean,TE_reference
0,16191,1587,Ring 2,Ring 2 was from,Japan,Canada,1.947800,1.503586
1,18648,511,"Portobello, Edinburgh","Portobello, Edinburgh is located in",Edinburgh,Victoria,5.278339,5.709949
2,13968,1360,Mariano Gonzalvo,Mariano Gonzalvo speaks,Spanish,Hebrew,6.031425,2.677353
3,6913,98,Alain de Benoist,Alain de Benoist speaks,French,Sanskrit,8.540665,2.980087
4,16682,2169,Black Sparrow Books,Black Sparrow Books is based in,Boston,Seoul,3.243492,1.306099


In [12]:
# SAVE THE PHASE 6B PROTOCOL BEFORE EVALUATION

protocol_v2 = {
    "experiment": "Phase 6B locked subject-disjoint replication",
    "model": MODEL_ID,
    "dataset": "azhx/counterfact:test",
    "N_fresh": N_FRESH,
    "sampling_seed": SAMPLING_SEED,
    "subject_disjoint_from_previous_phases": True,
    "unique_subject_within_phase6b": True,
    "excluded_phase5_subjects": len(PHASE5_SUBJECTS),
    "excluded_phase6a_subjects": len(PHASE6A_SUBJECTS),
    "total_excluded_subjects": len(EXCLUDED_SUBJECTS),
    "manifest_sha256": MANIFEST_SHA256,
    "AIE_pair": AIE_PAIR,
    "interaction_pair": INTERACTION_PAIR,
    "primary_scale": PRIMARY_SCALE,
    "noise_scales": NOISE_SCALES,
    "noise_seeds": NOISE_SEEDS,
    "reference_scale": REFERENCE_SCALE,
    "reference_seed": REFERENCE_SEED,
    "min_clean_margin": MIN_CLEAN_MARGIN,
    "min_reference_TE": MIN_REFERENCE_TE,
    "min_eval_TE": MIN_EVAL_TE,
    "primary_metric": "normalized causal recovery",
    "primary_success": "paired bootstrap 95% CI lower bound > 0",
    "practical_threshold": 0.02,
}

with open(PROTOCOL_V2, "w", encoding="utf-8") as f:
    json.dump(protocol_v2, f, indent=2)

print("protocol:", PROTOCOL_V2)


protocol: phase6b_locked_protocol.json


## STOP HERE BEFORE EVALUATION

At this point the manifest and protocol are locked.

Recommended terminal commit:

```bash
git add phase6_publication_validation.ipynb phase6b_locked_manifest.csv phase6b_locked_protocol.json
git commit -m "Freeze subject-disjoint Phase 6B replication"
git push
```

Only after the commit, continue with the evaluation cells below.


In [13]:
def patched_margin(
    case,
    clean_states,
    corrupt_emb,
    sites
):
    patches_by_layer = {}

    for layer, role in sites:
        patches_by_layer.setdefault(layer, []).append(role)

    handles = []

    for layer, roles in patches_by_layer.items():

        def make_hook(layer_idx, roles_here):

            def hook(module, inputs, output):
                if isinstance(output, tuple):
                    hidden = output[0].clone()
                else:
                    hidden = output.clone()

                for role in roles_here:
                    if role == "S":
                        pos = case["subject_last"]
                    elif role == "Q":
                        pos = case["query_pos"]
                    else:
                        raise ValueError(role)

                    clean_value = (
                        clean_states[layer_idx + 1][:, pos, :]
                    )

                    hidden[:, pos, :] = clean_value

                if isinstance(output, tuple):
                    return (hidden, *output[1:])

                return hidden

            return hook

        handle = model.transformer.h[layer].register_forward_hook(
            make_hook(layer, roles)
        )
        handles.append(handle)

    try:
        with torch.inference_mode():
            out = model(
                inputs_embeds=corrupt_emb,
                attention_mask=case["attention_mask"]
            )

        return logit_margin(
            out.logits,
            case["true_id"],
            case["new_id"]
        )

    finally:
        for handle in handles:
            handle.remove()


In [14]:
def evaluate_case_condition(
    case,
    noise_multiplier,
    noise_seed
):
    with torch.inference_mode():
        clean_out = model(
            input_ids=case["input_ids"],
            attention_mask=case["attention_mask"],
            output_hidden_states=True
        )

    clean_states = clean_out.hidden_states

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"]
    )

    actual_seed = (
        int(noise_seed)
        + int(case["case_id"])
    )

    corrupt_emb = corrupted_embeddings(
        case["input_ids"],
        case["subject_positions"],
        actual_seed,
        noise_multiplier
    )

    with torch.inference_mode():
        corrupt_out = model(
            inputs_embeds=corrupt_emb,
            attention_mask=case["attention_mask"]
        )

    D_corr = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"]
    )

    TE = D_clean - D_corr

    D_AIE = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        AIE_PAIR
    )

    D_INT = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        INTERACTION_PAIR
    )

    valid = TE >= MIN_EVAL_TE

    if valid:
        rec_AIE = (D_AIE - D_corr) / TE
        rec_INT = (D_INT - D_corr) / TE
        delta_rec = rec_INT - rec_AIE
    else:
        rec_AIE = np.nan
        rec_INT = np.nan
        delta_rec = np.nan

    return {
        "case_id": case["case_id"],
        "noise_scale": noise_multiplier,
        "noise_seed": noise_seed,
        "D_clean": D_clean,
        "D_corr": D_corr,
        "TE": TE,
        "valid": valid,
        "D_AIE": D_AIE,
        "D_INT": D_INT,
        "recovery_AIE": rec_AIE,
        "recovery_INT": rec_INT,
        "delta_recovery": delta_rec,
        "correct_AIE": float(D_AIE > 0),
        "correct_INT": float(D_INT > 0),
        "margin_advantage_INT": D_INT - D_AIE,
        "manifest_sha256": MANIFEST_SHA256,
    }


In [15]:
# SAFE RESUME
# If a result file exists, it must belong to THIS exact manifest.

if os.path.exists(RESULT_PATH):
    phase6b_results = pd.read_csv(RESULT_PATH)

    hashes = set(
        phase6b_results["manifest_sha256"].dropna().astype(str)
    )

    if hashes != {MANIFEST_SHA256}:
        raise RuntimeError(
            "Existing Phase-6B results do not match the current locked manifest."
        )
else:
    phase6b_results = pd.DataFrame()

done = set()

if len(phase6b_results) > 0:
    done = set(
        zip(
            phase6b_results["case_id"],
            phase6b_results["noise_scale"],
            phase6b_results["noise_seed"]
        )
    )

print("already completed conditions:", len(done))


already completed conditions: 0


In [16]:
# RUN THE LOCKED BENCHMARK WITH PROGRESSIVE CHECKPOINTING

rows = (
    phase6b_results.to_dict("records")
    if len(phase6b_results)
    else []
)

for ci, case in enumerate(fresh_cases_v2):

    for scale in NOISE_SCALES:
        for seed in NOISE_SEEDS:

            key = (
                case["case_id"],
                scale,
                seed
            )

            if key in done:
                continue

            row = evaluate_case_condition(
                case,
                scale,
                seed
            )

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(
                RESULT_PATH,
                index=False
            )

    print(
        f"{ci+1}/{len(fresh_cases_v2)}",
        case["subject"]
    )

phase6b_results = pd.DataFrame(rows)

expected_rows = N_FRESH * len(NOISE_SCALES) * len(NOISE_SEEDS)
print("rows:", len(phase6b_results), "/", expected_rows)

assert len(phase6b_results) == expected_rows
assert phase6b_results["case_id"].nunique() == N_FRESH


1/128 Ring 2
2/128 Portobello, Edinburgh
3/128 Mariano Gonzalvo
4/128 Alain de Benoist
5/128 Black Sparrow Books
6/128 Margaret Walker
7/128 Jimmy Choo
8/128 Carnival of Venice
9/128 Bavaria
10/128 Guillaumes
11/128 Johann Andreas Schmeller
12/128 Obsidian Entertainment
13/128 Il Gazzettino
14/128 Alain Mabanckou
15/128 Jules Paul Benjamin Delessert
16/128 Stevie Wonder
17/128 Sous le vent
18/128 Berlin International Film Festival
19/128 Dmitri Nabokov
20/128 Claudio Abbado
21/128 Acura Legend
22/128 Jean-Michel Jarre
23/128 Heilmann & Littmann
24/128 Autonomous University of Barcelona
25/128 Vadnagar
26/128 2011 Cannes Film Festival
27/128 Ouest-France
28/128 Honinbo Shuei
29/128 BC Hydro
30/128 Pelagius II
31/128 Irina Khakamada
32/128 Rudolf Kastner
33/128 Atul Chitnis
34/128 Ronaldinho
35/128 Italian Fascism
36/128 Tel Aviv
37/128 Adrian Ludwig Richter
38/128 Philip V of Spain
39/128 Bernadette Lafont
40/128 Vayots Dzor Region
41/128 Cadillac Sixty Special
42/128 Abdur Raheem Green

In [19]:
# DIAGNOSTICS

print(
    phase6b_results["valid"].value_counts()
)

print()

print(
    phase6b_results.groupby("noise_scale")[
        [
            "TE",
            "recovery_AIE",
            "recovery_INT",
            "delta_recovery"
        ]
    ].mean()
)


valid
True     1089
False      63
Name: count, dtype: int64

                   TE  recovery_AIE  recovery_INT  delta_recovery
noise_scale                                                      
2.5          3.066555      0.806440      0.853352        0.046912
3.0          3.177035      0.801122      0.847019        0.045897
3.5          3.245280      0.790298      0.837923        0.047626


In [20]:
# PRIMARY ENDPOINT: scale 3.0
# Average valid noise seeds within each factual case.
# The independent bootstrap unit is the case, not each corruption run.

primary = phase6b_results[
    (phase6b_results["noise_scale"] == PRIMARY_SCALE)
    & (phase6b_results["valid"])
].copy()

primary_case = (
    primary
    .groupby("case_id")
    .agg(
        recovery_AIE=("recovery_AIE", "mean"),
        recovery_INT=("recovery_INT", "mean"),
        delta=("delta_recovery", "mean"),
        correct_AIE=("correct_AIE", "mean"),
        correct_INT=("correct_INT", "mean"),
        n_valid_seeds=("noise_seed", "nunique"),
    )
)

missing_primary = (
    set(fresh_manifest_v2["case_id"])
    - set(primary_case.index)
)

print("cases in primary analysis:", len(primary_case))
print("cases with zero valid primary seeds:", len(missing_primary))
print("valid-seed counts:")
print(primary_case["n_valid_seeds"].value_counts().sort_index())

print()
print("Mean AIE recovery:", primary_case["recovery_AIE"].mean())
print("Mean interaction recovery:", primary_case["recovery_INT"].mean())
print("Mean delta:", primary_case["delta"].mean())


cases in primary analysis: 125
cases with zero valid primary seeds: 3
valid-seed counts:
n_valid_seeds
1      1
2      8
3    116
Name: count, dtype: int64

Mean AIE recovery: 0.8008758116785891
Mean interaction recovery: 0.8472835111262036
Mean delta: 0.04640769944761459


In [21]:
def bootstrap_mean_ci(
    values,
    n_boot=10000,
    seed=12345
):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    rng = np.random.default_rng(seed)
    n = len(x)
    boot = np.empty(n_boot)

    for b in range(n_boot):
        sample = rng.choice(
            x,
            size=n,
            replace=True
        )
        boot[b] = sample.mean()

    return (
        x.mean(),
        np.quantile(boot, 0.025),
        np.quantile(boot, 0.975)
    )


delta_mean, ci_lo, ci_hi = bootstrap_mean_ci(
    primary_case["delta"]
)

print("PRIMARY RESULT")
print("delta mean:", delta_mean)
print("95% CI:", ci_lo, ci_hi)
print("statistical success:", ci_lo > 0)
print("practical success:", delta_mean >= 0.02)


PRIMARY RESULT
delta mean: 0.04640769944761459
95% CI: 0.035520836795165954 0.05914597447675518
statistical success: True
practical success: True


In [22]:
# ROBUSTNESS ACROSS NOISE SCALES

robust_rows = []

for scale in NOISE_SCALES:

    d = phase6b_results[
        (phase6b_results["noise_scale"] == scale)
        & (phase6b_results["valid"])
    ]

    by_case = (
        d.groupby("case_id")["delta_recovery"]
        .mean()
    )

    mean_d, lo, hi = bootstrap_mean_ci(
        by_case.values
    )

    robust_rows.append({
        "scale": scale,
        "n_cases": len(by_case),
        "delta_mean": mean_d,
        "CI_lo": lo,
        "CI_hi": hi,
        "positive": lo > 0,
    })

robustness = pd.DataFrame(robust_rows)
print(robustness)


   scale  n_cases  delta_mean     CI_lo     CI_hi  positive
0    2.5      125    0.046882  0.036021  0.058332      True
1    3.0      125    0.046408  0.035521  0.059146      True
2    3.5      126    0.048872  0.036809  0.062691      True


In [23]:
# SECONDARY, HUMAN-READABLE SUMMARY

win_rate = (
    primary_case["delta"] > 0
).mean()

tie_rate = np.isclose(
    primary_case["delta"],
    0
).mean()

print("interaction win rate:", win_rate)
print("tie rate:", tie_rate)

summary = pd.DataFrame([{
    "n_primary_cases": len(primary_case),
    "mean_recovery_AIE": primary_case["recovery_AIE"].mean(),
    "mean_recovery_INT": primary_case["recovery_INT"].mean(),
    "delta_mean": delta_mean,
    "CI_lo": ci_lo,
    "CI_hi": ci_hi,
    "statistical_success": bool(ci_lo > 0),
    "practical_success": bool(delta_mean >= 0.02),
    "win_rate": win_rate,
    "tie_rate": tie_rate,
}])

summary.to_csv(
    "phase6b_primary_summary.csv",
    index=False
)

robustness.to_csv(
    "phase6b_robustness_summary.csv",
    index=False
)

summary


interaction win rate: 0.936
tie rate: 0.0


,n_primary_cases,mean_recovery_AIE,mean_recovery_INT,delta_mean,CI_lo,CI_hi,statistical_success,practical_success,win_rate,tie_rate
0,125,0.800876,0.847284,0.046408,0.035521,0.059146,True,True,0.936,0.0


## Interpretation rule

Treat Phase 6B as the decisive subject-disjoint replication.

- Do **not** change the two circuits after seeing these results.
- Do **not** search for a better noise seed if the result weakens.
- Report the number of invalid corruption conditions and any cases with zero valid primary seeds.
- Keep Phase 6A as an audited preliminary validation, not as the final untouched holdout.


In [24]:
complete_ids = primary_case[
    primary_case["n_valid_seeds"] == 3
].index

complete = primary_case.loc[
    complete_ids
]

mean3, lo3, hi3 = bootstrap_mean_ci(
    complete["delta"]
)

print("Complete 3-seed cases:", len(complete))
print("delta:", mean3)
print("95% CI:", lo3, hi3)

Complete 3-seed cases: 116
delta: 0.04561287501608928
95% CI: 0.03455129647437919 0.058822485271891975


In [25]:
raw_primary = phase6b_results[
    phase6b_results["noise_scale"] == PRIMARY_SCALE
]

raw_by_case = (
    raw_primary
    .groupby("case_id")["margin_advantage_INT"]
    .mean()
)

raw_mean, raw_lo, raw_hi = bootstrap_mean_ci(
    raw_by_case
)

print("Raw-margin cases:", len(raw_by_case))
print("mean margin advantage:", raw_mean)
print("95% CI:", raw_lo, raw_hi)

Raw-margin cases: 128
mean margin advantage: 0.12490370869636536
95% CI: 0.10278647045294442 0.1494302650292714
